In [18]:
import numpy as np
import pandas as pd
import joblib
import os

from scipy.stats import skew, kurtosis
df = pd.read_csv("gearbox_features.csv")

In [19]:
model = joblib.load("gearbox_model.pkl")

scaler = joblib.load("scaler.pkl")

encoder = joblib.load("label_encoder.pkl")

In [20]:
def extract_features(signal):

    features = {}

    num_sensors = signal.shape[1]

    for i in range(num_sensors):

        sensor = signal[:, i]

        features[f"S{i+1}_Mean"] = np.mean(sensor)
        features[f"S{i+1}_Median"] = np.median(sensor)
        features[f"S{i+1}_Std"] = np.std(sensor)
        features[f"S{i+1}_Variance"] = np.var(sensor)
        features[f"S{i+1}_Minimum"] = np.min(sensor)
        features[f"S{i+1}_Maximum"] = np.max(sensor)
        features[f"S{i+1}_Range"] = np.ptp(sensor)
        features[f"S{i+1}_RMS"] = np.sqrt(np.mean(sensor**2))
        features[f"S{i+1}_Peak"] = np.max(np.abs(sensor))
        features[f"S{i+1}_Peak_to_Peak"] = np.ptp(sensor)
        features[f"S{i+1}_Skewness"] = skew(sensor)
        features[f"S{i+1}_Kurtosis"] = kurtosis(sensor)
        features[f"S{i+1}_Energy"] = np.sum(sensor**2)

    return features

In [35]:
healthy_path = "Healthy Data"
broken_path = "BrokenTooth Data"

healthy_files = sorted(os.listdir(healthy_path))
broken_files = sorted(os.listdir(broken_path))

In [36]:
test_file = os.path.join(
    broken_path,
    "b30hz0.txt"
)

In [37]:
signal = np.loadtxt(test_file)

print("Signal shape:", signal.shape)

Signal shape: (88320, 4)


In [38]:
features = extract_features(signal)

new_data = pd.DataFrame([features])

In [39]:
X = df.drop(columns=["File", "Label"])

In [40]:
new_data = new_data[X.columns]

In [41]:
new_data_scaled = scaler.transform(new_data)

In [42]:
prediction = model.predict(new_data_scaled)

predicted_label = encoder.inverse_transform(prediction)

print("Prediction:", predicted_label[0])

Prediction: Broken


In [43]:
healthy_files = sorted([
    f for f in os.listdir(healthy_path)
    if f.endswith(".txt")
])

broken_files = sorted([
    f for f in os.listdir(broken_path)
    if f.endswith(".txt")
])

print("Healthy files:", len(healthy_files))
print("Broken files:", len(broken_files))

Healthy files: 10
Broken files: 10


In [45]:
results = []

for file in healthy_files:

    file_path = os.path.join(healthy_path, file)

    signal = np.loadtxt(file_path)

    features = extract_features(signal)

    new_data = pd.DataFrame([features])

    new_data = new_data[X.columns]

    new_data_scaled = scaler.transform(new_data)

    prediction = model.predict(new_data_scaled)

    predicted_label = encoder.inverse_transform(prediction)[0]

    results.append({
        "File": file,
        "Actual": "Healthy",
        "Predicted": predicted_label
    })


for file in broken_files:

    file_path = os.path.join(broken_path, file)

    signal = np.loadtxt(file_path)

    features = extract_features(signal)

    new_data = pd.DataFrame([features])

    new_data = new_data[X.columns]

    new_data_scaled = scaler.transform(new_data)

    prediction = model.predict(new_data_scaled)

    predicted_label = encoder.inverse_transform(prediction)[0]

    results.append({
        "File": file,
        "Actual": "Broken",
        "Predicted": predicted_label
    })

In [46]:
results_df = pd.DataFrame(results)

results_df

,File,Actual,Predicted
0,h30hz0.txt,Healthy,Healthy
1,h30hz10.txt,Healthy,Healthy
2,h30hz20.txt,Healthy,Healthy
3,h30hz30.txt,Healthy,Healthy
4,h30hz40.txt,Healthy,Healthy
5,h30hz50.txt,Healthy,Healthy
6,h30hz60.txt,Healthy,Healthy
7,h30hz70.txt,Healthy,Healthy
8,h30hz80.txt,Healthy,Healthy
9,h30hz90.txt,Healthy,Healthy


In [47]:
results_df["Correct"] = (
    results_df["Actual"] == results_df["Predicted"]
)

results_df

,File,Actual,Predicted,Correct
0,h30hz0.txt,Healthy,Healthy,True
1,h30hz10.txt,Healthy,Healthy,True
2,h30hz20.txt,Healthy,Healthy,True
3,h30hz30.txt,Healthy,Healthy,True
4,h30hz40.txt,Healthy,Healthy,True
5,h30hz50.txt,Healthy,Healthy,True
6,h30hz60.txt,Healthy,Healthy,True
7,h30hz70.txt,Healthy,Healthy,True
8,h30hz80.txt,Healthy,Healthy,True
9,h30hz90.txt,Healthy,Healthy,True


In [48]:
results_df.to_csv(
    "prediction_results.csv",
    index=False
)